# Exercise 01 — SOLUTION

**Try the stub first!** Only open this after you have attempted [ex01_stub.ipynb](ex01_stub.ipynb).

---

In [ ]:
!nvidia-smi

In [ ]:
%%writefile compute_current_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                        \
    cudaError_t e = (call);                                          \
    if (e != cudaSuccess) {                                          \
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(e)); \
        exit(1); }                                                   \
} while(0)

// ── SOLUTION: Kernel ──────────────────────────────────────────────────────────
__global__ void compute_input_current(
    const float* W, const float* X, float* I, float I_bias, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // global index
    if (i >= N) return;                                // bounds check
    I[i] = I_bias + W[i] * X[i];                      // weighted input
}

// ── SOLUTION: Challenge kernel ───────────────────────────────────────────────
__global__ void update_voltage(
    float* V, const float* I,
    float E_L, float Rm, float tau_m, float dt, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    // Euler step: dV/dt = (-(V - E_L) + Rm * I) / tau_m
    float dV = (-(V[i] - E_L) + Rm * I[i]) / tau_m;
    V[i] += dt * dV;
}

void compute_current_cpu(const float* W, const float* X, float* I,
                          float I_bias, int N) {
    for (int i = 0; i < N; i++) I[i] = I_bias + W[i] * X[i];
}

int main() {
    const int N = 50000;
    const float I_bias = 0.5f;
    const float E_L   = -65.0f;   // mV
    const float Rm    = 10.0f;    // MOhm
    const float tau_m = 20.0f;    // ms
    const float dt    = 0.1f;     // ms
    size_t bytes = N * sizeof(float);

    // Host
    float* h_W   = (float*)malloc(bytes);
    float* h_X   = (float*)malloc(bytes);
    float* h_I   = (float*)malloc(bytes);
    float* h_Iref= (float*)malloc(bytes);
    float* h_V   = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        h_W[i] = 0.1f * (i % 10);
        h_X[i] = (float)(i % 100) / 100.0f;
        h_V[i] = -65.0f;   // start at rest
    }

    // Device — allocate and copy
    float *d_W, *d_X, *d_I, *d_V;
    CUDA_CHECK(cudaMalloc(&d_W, bytes));
    CUDA_CHECK(cudaMalloc(&d_X, bytes));
    CUDA_CHECK(cudaMalloc(&d_I, bytes));
    CUDA_CHECK(cudaMalloc(&d_V, bytes));
    CUDA_CHECK(cudaMemcpy(d_W, h_W, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_X, h_X, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, bytes, cudaMemcpyHostToDevice));

    int threads = 256;
    int blocks  = (N + threads - 1) / threads;

    // ── Correctness check ─────────────────────────────────────────────────────
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    compute_input_current<<<blocks, threads>>>(d_W, d_X, d_I, I_bias, N);

    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    float ms; CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    CUDA_CHECK(cudaMemcpy(h_I, d_I, bytes, cudaMemcpyDeviceToHost));
    compute_current_cpu(h_W, h_X, h_Iref, I_bias, N);

    float max_err = 0.0f;
    for (int i = 0; i < N; i++) {
        float err = fabsf(h_I[i] - h_Iref[i]);
        if (err > max_err) max_err = err;
    }

    double bw = 3.0 * bytes / (ms * 1e-3) / 1e9;
    printf("=== compute_input_current ===\n");
    printf("N=%d  time=%.3f ms  BW=%.1f GB/s  max_err=%.2e  %s\n",
           N, ms, bw, max_err, max_err < 1e-5f ? "PASS" : "FAIL");

    // ── Challenge: 1000-step voltage update ───────────────────────────────────
    printf("\n=== update_voltage (1000 steps) ===\n");
    printf("Step     V[0] (mV)\n");
    printf("------   ---------\n");
    printf("%-6d   %.4f\n", 0, h_V[0]);

    for (int t = 1; t <= 1000; t++) {
        compute_input_current<<<blocks, threads>>>(d_W, d_X, d_I, I_bias, N);
        update_voltage<<<blocks, threads>>>(d_V, d_I, E_L, Rm, tau_m, dt, N);
        if (t % 100 == 0) {
            CUDA_CHECK(cudaMemcpy(h_V, d_V, bytes, cudaMemcpyDeviceToHost));
            printf("%-6d   %.4f\n", t, h_V[0]);
        }
    }

    cudaFree(d_W); cudaFree(d_X); cudaFree(d_I); cudaFree(d_V);
    free(h_W); free(h_X); free(h_I); free(h_Iref); free(h_V);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    return 0;
}

In [ ]:
!nvcc -O2 -o compute_current_sol compute_current_sol.cu -lm && ./compute_current_sol

## Reflection Answers

**1. What does `if (i >= N) return;` prevent?**  
The last block may have threads beyond index N-1. Without the guard, those threads would access `W[i]`, `X[i]`, and `I[i]` out of bounds — writing to unallocated GPU memory, causing undefined behavior or a CUDA error.

**2. What happens with different block sizes?**  
- 32 threads/block: valid (multiple of warp size 32), but poor SM occupancy. More blocks = more scheduling overhead.
- 1024 threads/block: valid (max allowed), but may limit occupancy if register/shared memory use is high.
- 1025 threads/block: `cudaErrorInvalidValue` — exceeds hardware maximum.

**3. Bytes per neuron?**  
Reads: W[i] (4 bytes) + X[i] (4 bytes) = 8 bytes read.  
Writes: I[i] (4 bytes) = 4 bytes written.  
Total: 12 bytes per neuron. For N=50,000: 600 KB total — tiny, so kernel time is dominated by launch overhead.

**4. Why no `cudaDeviceSynchronize` between kernels?**  
CUDA kernels launched on the same stream (the default stream) execute in order. The GPU guarantees that `update_voltage` will not start until `compute_input_current` is complete. Synchronization is only needed when the CPU needs to read GPU results.